In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, load_npz, csr_matrix
from sklearn.svm import LinearSVC
from sklearn.model_selection import cross_val_score, KFold
from xgboost import XGBClassifier
import numpy as np
from tqdm import tqdm
import pickle
import gzip
import reimport os


In [35]:
def readGz(path):
    for l in gzip.open(path, 'rt'):
        yield eval(l)

def readCSV(path):
    f = gzip.open(path, 'rt')
    f.readline()
    for l in f:
        yield l.strip().split(',')

def clean(t):
    t = t.lower()
    t = re.sub(r"[^a-z0-9\s']", " ", t)
    t = re.sub(r"\s+", " ", t)
    return t.strip()

print("Loading training data...")
data = [d for d in tqdm(readGz("train_Category.json.gz"), desc="Reading data")]
print("Cleaning texts...")
texts = [clean(d["review_text"]) for d in tqdm(data, desc="Cleaning")]
y = [d["genreID"] for d in data]
print(f"Loaded {len(data)} training examples")

Loading training data...


Reading data: 100000it [00:11, 8775.22it/s]


Extracting raw texts (before cleaning)...
Cleaning texts...


Cleaning: 100%|██████████| 100000/100000 [00:08<00:00, 11493.08it/s]


Loaded 100000 training examples


In [ ]:
word_vec = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,2),
    min_df=2,
    stop_words="english"
)
char_vec = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3,5),
    min_df=2,
    max_features=30000
)
    
def load_text_features(Xw_file="Xw_features.npz", Xc_file="Xc_features.npz"):
    if os.path.exists(Xw_file) and os.path.exists(Xc_file):
        print("Loading saved vectorized features...")
        Xw = load_npz(Xw_file)
        Xc = load_npz(Xc_file)
        print(f"Loaded Xw shape: {Xw.shape}, Xc shape: {Xc.shape}")
    else:
        print("Computing TF-IDF features (this may take a while)...")
        print("  Fitting word vectorizer...")
        Xw = word_vec.fit_transform(texts)
        print("  Fitting character vectorizer...")
        Xc = char_vec.fit_transform(texts)
        print("  Features computed!")
    return Xw, Xc

Loading cached TF-IDF features...
Fitting TF-IDF vectorizer (for transform)...
Loaded TF-IDF features shape: (100000, 100000)
Extracting enhanced features...


Extracting enhanced features: 100%|██████████| 100000/100000 [00:19<00:00, 5181.98it/s]


Enhanced features shape: (100000, 20)
Combined features shape: (100000, 100020)


In [37]:
Xw, Xc = load_text_features()

# Text features only
print("Combining text features...")
X_text = hstack([Xw, Xc])

print(f"Text features shape: {X_text.shape}")

# Convert y to numpy array for cross-validation
y = np.array(y)


Model Testing on 10k Examples
Preparing different feature combinations...
Testing on 10000 samples...
  tfidf_only: shape (10000, 100000)
  tfidf_basic_text: shape (10000, 100007)
  tfidf_rating_votes: shape (10000, 100002)
  tfidf_all_enhanced: shape (10000, 100020)

Training baseline SVM on 10k subset (TF-IDF only)...


Baseline CV: 100%|██████████| 5/5 [00:01<00:00,  3.96it/s]


Baseline (10k subset) CV Scores: [0.651, 0.672, 0.666, 0.658, 0.6905]
Baseline Mean: 0.6675 (+/- 0.0270)

Testing Feature Combinations and Hyperparameters on 10k Subset

Testing 48 combinations (4 feature sets × 12 param sets)...


KeyboardInterrupt: 

In [ ]:
# Train final XGBoost model on all data
print("\nTraining final XGBoost model on all data...")
mod_final = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    tree_method='hist',
    n_jobs=-1,
    verbosity=1
)
mod_final.fit(X_text, y)
print("Model trained!")

# Make predictions on test set
predictions = open("predictions_Category.csv", "w")
predictions.write("userID,reviewID,prediction\n")

print("Making predictions on test set...")
for l in tqdm(readGz("test_Category.json.gz"), desc="Writing predictions"):
    text = clean(l["review_text"])
    
    # Text features
    Xw = word_vec.transform([text])
    Xc = char_vec.transform([text])
    Xtest = hstack([Xw, Xc])

    pred = mod_final.predict(Xtest)[0]
    predictions.write(f"{l['user_id']},{l['review_id']},{pred}\n")

predictions.close()
print("Predictions saved to predictions_Category.csv")


Making predictions on test set...


Writing predictions: 10000it [00:50, 198.11it/s]

Predictions saved to predictions_Category.csv
